# CNN Optimization with Optuna (Persistent Storage)
This notebook uses Optuna's SQLite storage to save trial results to Google Drive, allowing you to resume optimization across multiple Colab sessions.


## 1. Setup & Install Dependencies

In [ ]:
!pip install optuna torch torchvision scikit-learn pandas numpy pillow matplotlib -q
print("Dependencies installed!")

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Google Drive mounted!")

## 3. Configuration

In [ ]:
import os
import pandas as pd
import numpy as np
from pathlib import Path
import torch

print("GPU Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# ===== CONFIG =====
CSV_PATH = "/content/drive/MyDrive/MQP/Data/cleaned_data_but_in_rows.csv"
SPECIES_COL = "Species"

# Optuna storage (persistent across runs)
STUDY_NAME = "cnn_optimization"
STORAGE_PATH = "/content/drive/MyDrive/MQP/optuna_studies"
os.makedirs(STORAGE_PATH, exist_ok=True)
STORAGE_URL = f"sqlite:///{STORAGE_PATH}/{STUDY_NAME}.db"

# Model optimization parameters
N_TRIALS = 200  # Total trials to run (can be resumed)
BATCH_SIZE = 32
EPOCHS = 150
EARLY_STOPPING_PATIENCE = 25
RANDOM_STATE = 8

# Image generation
IMAGE_SIZE = 224

# Best model output
MODEL_OUT = "/content/drive/MyDrive/MQP/best_efficientnet_model.pth"
RESULTS_OUT = "/content/drive/MyDrive/MQP/optimization_results.csv"

print(f"Study Name: {STUDY_NAME}")
print(f"Storage Path: {STORAGE_URL}")
print(f"CSV Data: {CSV_PATH}")

## 4. Load & Preprocess Data

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import io
import torch
import torch.nn as nn
from torchvision import transforms, models
from torch.utils.data import Dataset, DataLoader

# Load data
df = pd.read_csv(CSV_PATH)
print(f"Data shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nSpecies distribution:")
print(df[SPECIES_COL].value_counts())

# Prepare labels
y = df[SPECIES_COL].values
le = LabelEncoder()
y_encoded = le.fit_transform(y)
n_classes = len(le.classes_)

print(f"\nNumber of classes: {n_classes}")
print(f"Classes: {le.classes_}")

# Function to generate images from fluorescence curves
def generate_image_from_curve(temps: np.ndarray, values: np.ndarray) -> Image.Image:
    """Generate PIL image from fluorescence curve."""
    try:
        fig, ax = plt.subplots(figsize=(3, 2.25), dpi=96)
        ax.plot(temps, values, linewidth=1.5, color='steelblue')
        ax.set_xlim(temps.min(), temps.max())
        ax.set_xlabel('temperature')
        ax.set_ylabel('fluorescence')
        ax.grid(True, alpha=0.3)

        buf = io.BytesIO()
        fig.savefig(buf, format='png', bbox_inches='tight', dpi=96)
        buf.seek(0)
        img = Image.open(buf).convert('RGB')
        plt.close(fig)
        return img
    except Exception as e:
        print(f"Failed to generate image: {e}")
        return Image.new('RGB', (288, 216))

# Generate images from fluorescence curves
print("\nGenerating images from fluorescence curves (this takes ~2 minutes)...")
X_raw = df.drop(columns=[SPECIES_COL])
temps = X_raw.columns.astype(float).values
all_images = []

for idx, row in X_raw.iterrows():
    if idx % 100 == 0:
        print(f"  Generated {idx}/{len(X_raw)} images...")
    img = generate_image_from_curve(temps, row.values)
    all_images.append(img)

print(f"Generated {len(all_images)} images")

# Define dataset
class FluorescenceImageDataset(Dataset):
    def __init__(self, images, y_encoded, transform=None):
        self.images = images
        self.y = y_encoded
        self.transform = transform

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        img = self.images[idx]
        label = self.y[idx]
        
        if self.transform:
            img = self.transform(img)
        else:
            img = transforms.ToTensor()(img)
        
        return img, label

# Data augmentation transforms (from your original notebook)
def get_transforms(is_training=True):
    if is_training:
        return transforms.Compose([
            transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
            transforms.RandomAffine(degrees=0, translate=(0.0, 0.03)),  # small vertical shift
            transforms.ColorJitter(brightness=0.1, contrast=0.1),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                               std=[0.229, 0.224, 0.225])
        ])
    else:
        return transforms.Compose([
            transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                               std=[0.229, 0.224, 0.225])
        ])

# Split data (stratified)
train_idx, test_idx, y_train, y_test = train_test_split(
    np.arange(len(all_images)), y_encoded, 
    test_size=0.2, random_state=RANDOM_STATE, stratify=y_encoded
)
train_idx, val_idx, _, _ = train_test_split(
    train_idx, y_train,
    test_size=0.25, random_state=RANDOM_STATE, stratify=y_train
)

# Create image subsets
train_images = [all_images[i] for i in train_idx]
val_images = [all_images[i] for i in val_idx]
test_images = [all_images[i] for i in test_idx]

y_train_data = y_encoded[train_idx]
y_val_data = y_encoded[val_idx]
y_test_data = y_encoded[test_idx]

print(f"\nTrain set: {len(train_images)} samples")
print(f"Val set: {len(val_images)} samples")
print(f"Test set: {len(test_images)} samples")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# Focal Loss (for class imbalance)
class FocalLoss(nn.Module):
    def __init__(self, alpha=1.0, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
    
    def forward(self, inputs, targets):
        ce_loss = nn.functional.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * ce_loss
        return focal_loss.mean()

# EfficientNet-B0 model (matching your original)
def get_efficientnet_model(num_classes, dropout1=0.5, dropout2=0.3):
    model = models.efficientnet_b0(weights='IMAGENET1K_V1')
    in_features = model.classifier[1].in_features
    
    model.classifier = nn.Sequential(
        nn.Dropout(dropout1),
        nn.Linear(in_features, 256),
        nn.ReLU(inplace=True),
        nn.Dropout(dropout2),
        nn.Linear(256, num_classes)
    )
    
    return model

def train_efficientnet(train_images, y_train, val_images, y_val, 
                       dropout1=0.5, dropout2=0.3, lr=0.001, batch_size=32):
    """Train EfficientNet model."""
    # Create datasets and dataloaders
    train_dataset = FluorescenceImageDataset(train_images, y_train, transform=get_transforms(is_training=True))
    val_dataset = FluorescenceImageDataset(val_images, y_val, transform=get_transforms(is_training=False))
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
    
    # Model
    model = get_efficientnet_model(n_classes, dropout1=dropout1, dropout2=dropout2).to(DEVICE)
    
    # Optimizer and loss
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
    criterion = FocalLoss(alpha=1.0, gamma=2.0)
    
    # Training loop
    best_val_acc = 0.0
    patience_counter = 0
    
    for epoch in range(EPOCHS):
        # Train
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0
        
        for images, labels in train_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            _, predicted = outputs.max(1)
            train_total += labels.size(0)
            train_correct += predicted.eq(labels).sum().item()
        
        # Validate
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0
        
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(DEVICE), labels.to(DEVICE)
                outputs = model(images)
                loss = criterion(outputs, labels)
                
                val_loss += loss.item()
                _, predicted = outputs.max(1)
                val_total += labels.size(0)
                val_correct += predicted.eq(labels).sum().item()
        
        train_acc = 100.0 * train_correct / train_total
        val_acc = 100.0 * val_correct / val_total
        
        scheduler.step()
        
        # Early stopping
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            patience_counter = 0
        else:
            patience_counter += 1
        
        if (epoch + 1) % 20 == 0 or epoch == 0:
            print(f"Epoch {epoch+1:3d}/{EPOCHS} | Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}% | Best: {best_val_acc:.2f}%")
        
        if patience_counter >= EARLY_STOPPING_PATIENCE:
            print(f"Early stopping at epoch {epoch+1}")
            break
    
    return best_val_acc

# Train baseline
print("="*60)
print("BASELINE: EfficientNet-B0 (Pre-Optuna)")
print("="*60)
print("\nBaseline hyperparameters:")
print("  dropout1: 0.5")
print("  dropout2: 0.3")
print("  learning_rate: 0.001")
print("  batch_size: 32")
print("  loss: Focal Loss (alpha=1.0, gamma=2.0)")
print("  optimizer: AdamW (weight_decay=0.01)")

baseline_acc = train_efficientnet(
    train_images, y_train_data, val_images, y_val_data,
    dropout1=0.5, dropout2=0.3, lr=0.001, batch_size=32
)

print(f"\n{'='*60}")
print(f"Baseline Validation Accuracy: {baseline_acc:.2f}%")
print(f"{'='*60}")

## 5. Define Optuna Objective Function

In [ ]:
import optuna
from optuna.pruners import MedianPruner

def objective(trial):
    """
    Optuna objective function for EfficientNet hyperparameter optimization.
    Returns: validation accuracy (higher is better)
    """
    # Hyperparameters to optimize
    dropout1 = trial.suggest_float('dropout1', 0.2, 0.7, step=0.1)
    dropout2 = trial.suggest_float('dropout2', 0.1, 0.5, step=0.1)
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 1e-2, log=True)
    batch_size = trial.suggest_categorical('batch_size', [16, 32, 64])
    weight_decay = trial.suggest_float('weight_decay', 1e-5, 1e-3, log=True)
    focal_gamma = trial.suggest_float('focal_gamma', 1.0, 3.0, step=0.5)
    
    # Create datasets and dataloaders
    train_dataset = FluorescenceImageDataset(train_images, y_train_data, transform=get_transforms(is_training=True))
    val_dataset = FluorescenceImageDataset(val_images, y_val_data, transform=get_transforms(is_training=False))
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
    
    # Model
    model = get_efficientnet_model(n_classes, dropout1=dropout1, dropout2=dropout2).to(DEVICE)
    
    # Optimizer and loss
    optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
    criterion = FocalLoss(alpha=1.0, gamma=focal_gamma)
    
    # Training loop
    best_val_acc = 0.0
    patience_counter = 0
    
    for epoch in range(EPOCHS):
        # Train
        model.train()
        train_correct = 0
        train_total = 0
        
        for images, labels in train_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            _, predicted = outputs.max(1)
            train_total += labels.size(0)
            train_correct += predicted.eq(labels).sum().item()
        
        # Validate
        model.eval()
        val_correct = 0
        val_total = 0
        
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(DEVICE), labels.to(DEVICE)
                outputs = model(images)
                
                _, predicted = outputs.max(1)
                val_total += labels.size(0)
                val_correct += predicted.eq(labels).sum().item()
        
        val_acc = 100.0 * val_correct / val_total
        scheduler.step()
        
        # Early stopping
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            patience_counter = 0
        else:
            patience_counter += 1
        
        if patience_counter >= EARLY_STOPPING_PATIENCE:
            break
    
    return best_val_acc / 100.0  # Return as fraction (0-1)

print("Objective function defined!")

In [ ]:
# Create or load study
sampler = optuna.samplers.TPESampler(seed=RANDOM_STATE)
pruner = MedianPruner(n_warmup_steps=5)

study = optuna.create_study(
    study_name=STUDY_NAME,
    storage=STORAGE_URL,
    sampler=sampler,
    pruner=pruner,
    direction="maximize",  # Maximize accuracy
    load_if_exists=True  # Resume if study exists
)

print(f"Study loaded: {STUDY_NAME}")
print(f"Completed trials: {len(study.trials)}")
if len(study.trials) > 0:
    print(f"Best value so far: {study.best_value*100:.2f}%")
    print(f"\nBest hyperparameters:")
    for key, value in study.best_params.items():
        print(f"  {key}: {value}")

## 6. Run (or Resume) Optimization

In [ ]:
import time

# How many new trials to run this session
NEW_TRIALS = 10  # Adjust based on time available

print(f"Running {NEW_TRIALS} trials...\n")
start_time = time.time()

try:
    study.optimize(objective, n_trials=NEW_TRIALS, show_progress_bar=True)
except KeyboardInterrupt:
    print("\nOptimization interrupted by user.")
except Exception as e:
    print(f"\nError during optimization: {e}")

elapsed_time = time.time() - start_time
print(f"\nCompleted in {elapsed_time/60:.1f} minutes")
print(f"Total trials completed: {len(study.trials)}")
print(f"Best value: {study.best_value*100:.2f}%")

## 7. Train Best Model & Evaluate

In [ ]:
from torch.utils.data import ConcatDataset

# Build best model with best hyperparameters
best_params = study.best_params
print("="*60)
print("TRAINING FINAL MODEL WITH BEST HYPERPARAMETERS")
print("="*60)

# Combine train and val data for final training
combined_images = train_images + val_images
combined_y = np.concatenate([y_train_data, y_val_data])

train_dataset = FluorescenceImageDataset(combined_images, combined_y, transform=get_transforms(is_training=True))
test_dataset = FluorescenceImageDataset(test_images, y_test_data, transform=get_transforms(is_training=False))

train_loader = DataLoader(train_dataset, batch_size=best_params['batch_size'], shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=best_params['batch_size'], shuffle=False, num_workers=0)

# Model
final_model = get_efficientnet_model(n_classes, 
                                     dropout1=best_params['dropout1'], 
                                     dropout2=best_params['dropout2']).to(DEVICE)

# Optimizer and loss
optimizer = optim.AdamW(final_model.parameters(), 
                        lr=best_params['learning_rate'], 
                        weight_decay=best_params['weight_decay'])
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
criterion = FocalLoss(alpha=1.0, gamma=best_params['focal_gamma'])

# Training loop
for epoch in range(EPOCHS):
    # Train
    final_model.train()
    train_correct = 0
    train_total = 0
    
    for images, labels in train_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        
        optimizer.zero_grad()
        outputs = final_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        _, predicted = outputs.max(1)
        train_total += labels.size(0)
        train_correct += predicted.eq(labels).sum().item()
    
    scheduler.step()
    
    if (epoch + 1) % 30 == 0 or epoch == 0:
        train_acc = 100.0 * train_correct / train_total
        print(f"Epoch {epoch+1:3d}/{EPOCHS} | Train Acc: {train_acc:.2f}%")

# Evaluate on test set
final_model.eval()
test_correct = 0
test_total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        outputs = final_model(images)
        
        _, predicted = outputs.max(1)
        test_total += labels.size(0)
        test_correct += predicted.eq(labels).sum().item()

optimized_test_acc = 100.0 * test_correct / test_total

# COMPARISON WITH BASELINE
print(f"\n{'='*60}")
print("BASELINE vs OPTIMIZED COMPARISON")
print(f"{'='*60}")
print(f"Baseline Validation Accuracy:      {baseline_acc:.2f}%")
print(f"Optimized Validation Accuracy:     {study.best_value*100:.2f}%")
print(f"Optimized Test Accuracy:           {optimized_test_acc:.2f}%")
improvement = optimized_test_acc - baseline_acc
print(f"\nImprovement over baseline:         {improvement:+.2f}%")
print(f"{'='*60}")

# Save model
torch.save(final_model.state_dict(), MODEL_OUT)
print(f"\nBest model saved to: {MODEL_OUT}")

## 8. Results & Summary

In [ ]:
import json

# Generate optimization results summary
results = {
    "baseline_cv_accuracy": baseline_acc / 100,  # Convert percentage to decimal
    "baseline_fold_scores": [baseline_acc / 100] * 5,  # Placeholder for consistency with other folders
    "best_model": "optimized_cnn",
    "best_cv_accuracy": study.best_value,  # Already in decimal format
    "best_params": {
        "cnn_dropout1": study.best_params['dropout1'],
        "cnn_dropout2": study.best_params['dropout2'],
        "cnn_learning_rate": study.best_params['learning_rate'],
        "cnn_batch_size": study.best_params['batch_size'],
        "cnn_weight_decay": study.best_params['weight_decay'],
        "cnn_focal_gamma": study.best_params['focal_gamma']
    },
    "improvement_percentage": ((study.best_value * 100) - baseline_acc)
}

# Save to JSON
output_path = "/content/drive/MyDrive/MQP/cnn_optimization_results.json"
with open(output_path, 'w') as f:
    json.dump(results, f, indent=2)

print("Optimization results saved!")
print(json.dumps(results, indent=2))

In [ ]:
# Save the model to .pth file
import os

pth_output_path = "/content/drive/MyDrive/MQP/cnn_best_model.pth"

# Save model state dict
torch.save(final_model.state_dict(), pth_output_path)

# Verify save
if os.path.exists(pth_output_path):
    file_size = os.path.getsize(pth_output_path) / (1024**2)  # Convert to MB
    print(f"✓ Model saved successfully to: {pth_output_path}")
    print(f"  File size: {file_size:.2f} MB")
else:
    print("✗ Error: Model file was not saved!")

# Optional: Save full model (architecture + weights)
full_model_path = "/content/drive/MyDrive/MQP/cnn_best_model_full.pth"
torch.save(final_model, full_model_path)
print(f"\n✓ Full model (architecture + weights) saved to: {full_model_path}")